# Stage 9.5 — Raw Live Webcam OpenVINO Recognition

## Purpose

Stage 9.5 converts the validated offline recognition pipeline into a live
webcam recognition engine.

The frozen Stage 9.1 OpenVINO MobileNetV3-Large embedding extractor and the
canonical Stage 4.5 prototype gallery are retained without modification.

Unlike the offline isolated-sign pipeline, a webcam stream has no known sign
start or end. Therefore, Stage 9.5 uses a rolling temporal window of recent
frames.

---

## Live Recognition Pipeline

Webcam  
→ continuous frame capture  
→ temporal frame sampling  
→ rolling 8-frame embedding buffer  
→ OpenVINO MobileNetV3-Large  
→ 960-D frame embeddings  
→ mean temporal pooling  
→ L2 normalization  
→ cosine similarity against 4,764 frozen prototypes  
→ Top-5 raw gloss predictions  
→ live display

---

## Important Experimental Boundary

Stage 9.5 intentionally reports **raw recognition output**.

The following mechanisms are NOT applied yet:

- confidence-based rejection
- UNKNOWN prediction
- majority voting
- temporal voting
- prediction persistence
- duplicate suppression
- sentence generation
- TTS

These mechanisms are reserved for Stage 9.6 so that the behaviour of the raw
recognition system can be observed independently from stabilization.

A high cosine similarity must not automatically be interpreted as prediction
confidence or correctness. Similarity thresholds will therefore not be tuned
during Stage 9.5.

---

## Scientific / Methodological Disclaimer

No training, fine-tuning, prototype adaptation, test-label tuning, Hand-ROI
processing, TTA, whitening, or CSLS is performed.

The webcam user is not part of the frozen CISLR evaluation set, so live webcam
predictions are a deployment demonstration rather than an official recognition
accuracy measurement.

Stage 9.5 evaluates raw isolated-sign retrieval behaviour under live input.
It must not be described as continuous sign-language translation.

---

## Controls

- `Q` — stop webcam and finish Stage 9.5
- `S` — save the current webcam frame for qualitative audit

---

## Stage 9.5 Success Criteria

Stage 9.5 passes the engineering validation when:

1. The webcam opens successfully.
2. Frames are continuously captured.
3. OpenVINO inference operates on live frames.
4. A rolling 8-frame temporal representation is generated.
5. The frozen 4,764-prototype gallery is queried successfully.
6. Raw Top-5 predictions are displayed live.
7. Runtime latency and FPS are measured.
8. A deployment log and summary report are saved.
9. The application terminates cleanly without modifying frozen upstream artifacts.

Prediction stability and rejection are evaluated separately in Stage 9.6.

In [4]:
# ==================================================================================================
# CISLR — STAGE 9.5
# RAW LIVE WEBCAM OPENVINO RECOGNITION
# ==================================================================================================
#
# Frozen recognition:
#
# Webcam
#   -> sample live frames
#   -> OpenVINO MobileNetV3-Large
#   -> 960-D frame embeddings
#   -> rolling 8-embedding temporal buffer
#   -> mean pooling
#   -> L2 normalization
#   -> cosine retrieval against 4764 Stage 4.5 prototypes
#   -> raw Top-5 predictions
#
# IMPORTANT:
#   NO stabilization
#   NO UNKNOWN threshold
#   NO confidence tuning
#   NO majority voting
#   NO sentence generation
#   NO TTS
#
# Controls:
#   Q = quit
#   S = save screenshot
#
# ==================================================================================================

from pathlib import Path
from collections import deque
from datetime import datetime

import csv
import json
import time

import cv2
import numpy as np
import pandas as pd

from openvino import Core


# ==================================================================================================
# 0. CONFIGURATION
# ==================================================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

STAGE9_ROOT = (
    ROOT
    / "audit"
    / "stage9_final_openvino"
)

STAGE9_5_DIR = (
    STAGE9_ROOT
    / "realtime_validation"
)

LOG_DIR = (
    STAGE9_5_DIR
    / "logs"
)

REPORT_DIR = (
    STAGE9_5_DIR
    / "reports"
)

SCREENSHOT_DIR = (
    STAGE9_5_DIR
    / "screenshots"
)


for directory in [
    STAGE9_5_DIR,
    LOG_DIR,
    REPORT_DIR,
    SCREENSHOT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# --------------------------------------------------------------------------------------------------
# Frozen OpenVINO model
# --------------------------------------------------------------------------------------------------

OV_XML = (
    STAGE9_ROOT
    / "embedding_fp32"
    / "mobilenet_v3_large_embedding_960_fp32_static.xml"
)


# --------------------------------------------------------------------------------------------------
# Frozen canonical Stage 4.5 gallery
# --------------------------------------------------------------------------------------------------

STAGE45_EMBED_DIR = (
    ROOT
    / "audit"
    / "stage4_5_official_evaluation"
    / "embeddings"
)

PROTOTYPE_EMBED_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_embeddings.npy"
)

PROTOTYPE_META_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_metadata.csv"
)


# --------------------------------------------------------------------------------------------------
# Live deployment parameters
# --------------------------------------------------------------------------------------------------

CAMERA_INDEX = 0

TEMPORAL_WINDOW = 8

# Run the embedding model once every N captured webcam frames.
# Starting conservatively with every 3rd frame.
INFERENCE_EVERY = 3

TOP_K = 5

OPENVINO_DEVICE = "CPU"

REQUESTED_WIDTH = 1280
REQUESTED_HEIGHT = 720
REQUESTED_CAMERA_FPS = 30

WINDOW_NAME = (
    "CISLR Stage 9.5 - "
    "Raw OpenVINO Recognition"
)


print("=" * 100)
print("CISLR — STAGE 9.5")
print("RAW LIVE WEBCAM OPENVINO RECOGNITION")
print("=" * 100)


# ==================================================================================================
# 1. ARTIFACT CHECK
# ==================================================================================================

required_files = {
    "OpenVINO model":
        OV_XML,

    "Prototype embeddings":
        PROTOTYPE_EMBED_PATH,

    "Prototype metadata":
        PROTOTYPE_META_PATH,
}


print("\nARTIFACT CHECK")
print("-" * 100)


for name, path in required_files.items():

    print(
        f"{name:25s}: {path}"
    )

    print(
        f"{'':25s}  Exists = {path.exists()}"
    )


missing = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]


if missing:

    raise FileNotFoundError(
        "Missing Stage 9.5 artifacts:\n"
        + "\n".join(missing)
    )


print("\n[OK] Required artifacts exist.")


# ==================================================================================================
# 2. LOAD + NORMALIZE PROTOTYPE GALLERY
# ==================================================================================================

prototype_bank = np.load(
    PROTOTYPE_EMBED_PATH
).astype(np.float32)

prototype_meta = pd.read_csv(
    PROTOTYPE_META_PATH
)


if prototype_bank.ndim != 2:

    raise ValueError(
        f"Prototype bank must be 2-D. "
        f"Got {prototype_bank.shape}"
    )


if prototype_bank.shape[1] != 960:

    raise ValueError(
        "Expected 960-D embeddings. "
        f"Got {prototype_bank.shape[1]}"
    )


if len(prototype_bank) != len(
    prototype_meta
):

    raise ValueError(
        "Prototype bank / metadata mismatch."
    )


if not {
    "uid",
    "gloss"
}.issubset(
    prototype_meta.columns
):

    raise ValueError(
        "Prototype metadata must contain "
        "'uid' and 'gloss'."
    )


prototype_norms = np.linalg.norm(
    prototype_bank,
    axis=1,
    keepdims=True
)


prototype_bank = (
    prototype_bank
    / np.clip(
        prototype_norms,
        1e-12,
        None
    )
).astype(np.float32)


print("\n" + "=" * 100)
print("PROTOTYPE GALLERY")
print("=" * 100)

print(
    "Prototype count :",
    prototype_bank.shape[0]
)

print(
    "Embedding dim   :",
    prototype_bank.shape[1]
)

print(
    "Unique glosses  :",
    prototype_meta[
        "gloss"
    ].nunique()
)

print("\n[OK] Gallery ready.")


# ==================================================================================================
# 3. LOAD OPENVINO MODEL
# ==================================================================================================

core = Core()

ov_model = core.read_model(
    str(OV_XML)
)

compiled_model = core.compile_model(
    ov_model,
    OPENVINO_DEVICE
)

input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)


print("\n" + "=" * 100)
print("OPENVINO")
print("=" * 100)

print(
    "Device      :",
    OPENVINO_DEVICE
)

print(
    "Input shape :",
    tuple(
        input_layer.shape
    )
)

print(
    "Output shape:",
    tuple(
        output_layer.shape
    )
)


if tuple(
    input_layer.shape
) != (
    1,
    3,
    224,
    224
):

    raise ValueError(
        "Unexpected OpenVINO input shape."
    )


if tuple(
    output_layer.shape
) != (
    1,
    960
):

    raise ValueError(
        "Unexpected OpenVINO output shape."
    )


print("\n[OK] OpenVINO model ready.")


# ==================================================================================================
# 4. FROZEN IMAGE PREPROCESSING
# ==================================================================================================

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(
    1,
    1,
    3
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(
    1,
    1,
    3
)


def preprocess_live_frame(
    frame_bgr
):

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    frame_rgb = cv2.resize(
        frame_rgb,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )

    x = (
        frame_rgb.astype(
            np.float32
        )
        / 255.0
    )

    x = (
        x - IMAGENET_MEAN
    ) / IMAGENET_STD

    x = np.transpose(
        x,
        (2, 0, 1)
    )

    x = np.expand_dims(
        x,
        axis=0
    )

    return np.ascontiguousarray(
        x,
        dtype=np.float32
    )


# ==================================================================================================
# 5. LIVE FRAME EMBEDDING
# ==================================================================================================

def extract_live_embedding(
    frame_bgr
):

    x = preprocess_live_frame(
        frame_bgr
    )

    start = time.perf_counter()

    result = compiled_model(
        [x]
    )

    inference_ms = (
        time.perf_counter()
        - start
    ) * 1000.0


    embedding = np.asarray(
        result[
            output_layer
        ]
    ).reshape(-1)


    if embedding.shape != (
        960,
    ):

        raise ValueError(
            "Unexpected live embedding "
            f"shape: {embedding.shape}"
        )


    return (
        embedding.astype(
            np.float32
        ),
        float(
            inference_ms
        )
    )


# ==================================================================================================
# 6. TEMPORAL EMBEDDING
# ==================================================================================================

def make_temporal_embedding(
    embedding_buffer
):
    """
    Mean pool the most recent 8 spatial embeddings,
    then L2 normalize.
    """

    if len(
        embedding_buffer
    ) < TEMPORAL_WINDOW:

        return None


    stacked = np.stack(
        list(
            embedding_buffer
        ),
        axis=0
    )


    temporal_embedding = (
        stacked.mean(
            axis=0
        )
    )


    norm = float(
        np.linalg.norm(
            temporal_embedding
        )
    )


    if norm <= 1e-12:

        return None


    temporal_embedding = (
        temporal_embedding
        / norm
    ).astype(
        np.float32
    )


    return temporal_embedding


# ==================================================================================================
# 7. RAW TOP-K RETRIEVAL
# ==================================================================================================

def retrieve_live_topk(
    query_embedding,
    k=5
):

    start = time.perf_counter()


    similarities = (
        prototype_bank
        @ query_embedding
    )


    k = min(
        int(k),
        len(
            similarities
        )
    )


    top_indices = np.argpartition(
        -similarities,
        k - 1
    )[:k]


    top_indices = (
        top_indices[
            np.argsort(
                -similarities[
                    top_indices
                ]
            )
        ]
    )


    retrieval_ms = (
        time.perf_counter()
        - start
    ) * 1000.0


    predictions = []


    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        metadata = (
            prototype_meta.iloc[
                int(idx)
            ]
        )


        predictions.append(
            {
                "rank":
                    int(rank),

                "prototype_index":
                    int(idx),

                "prototype_uid":
                    str(
                        metadata[
                            "uid"
                        ]
                    ),

                "gloss":
                    str(
                        metadata[
                            "gloss"
                        ]
                    ),

                "similarity":
                    float(
                        similarities[
                            idx
                        ]
                    ),
            }
        )


    return (
        predictions,
        float(
            retrieval_ms
        )
    )


# ==================================================================================================
# 8. DISPLAY HELPER
# ==================================================================================================

def draw_text(
    image,
    text,
    position,
    scale=0.65,
    thickness=2
):

    cv2.putText(
        image,
        str(text),
        position,
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )


# ==================================================================================================
# 9. OPEN WEBCAM
# ==================================================================================================

print("\n" + "=" * 100)
print("OPENING WEBCAM")
print("=" * 100)


cap = cv2.VideoCapture(
    CAMERA_INDEX
)


if not cap.isOpened():

    raise RuntimeError(
        f"Could not open webcam "
        f"index {CAMERA_INDEX}."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    REQUESTED_WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    REQUESTED_HEIGHT
)

cap.set(
    cv2.CAP_PROP_FPS,
    REQUESTED_CAMERA_FPS
)


actual_width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

actual_height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

actual_camera_fps = float(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)


print(
    "Camera index :",
    CAMERA_INDEX
)

print(
    "Resolution   :",
    f"{actual_width} x "
    f"{actual_height}"
)

print(
    "Camera FPS   :",
    actual_camera_fps
)

print(
    "Inference every:",
    INFERENCE_EVERY,
    "captured frames"
)

print(
    "Temporal window:",
    TEMPORAL_WINDOW,
    "embeddings"
)

print("\nControls:")
print(" Q = quit")
print(" S = save screenshot")

print("\n[OK] Webcam opened.")


# ==================================================================================================
# 10. LIVE STATE
# ==================================================================================================

embedding_buffer = deque(
    maxlen=TEMPORAL_WINDOW
)


frame_counter = 0
prediction_counter = 0
screenshot_counter = 0


latest_predictions = []
latest_top1 = "WARMING UP"
latest_similarity = None


inference_latencies = []
retrieval_latencies = []
prediction_latencies = []
loop_fps_values = []


log_rows = []


session_start = (
    time.perf_counter()
)

last_loop_time = (
    time.perf_counter()
)


# ==================================================================================================
# 11. LIVE LOOP
# ==================================================================================================

print("\n" + "=" * 100)
print("LIVE RECOGNITION STARTED")
print("=" * 100)


try:

    while True:

        loop_start = (
            time.perf_counter()
        )


        ok, frame = cap.read()


        if not ok or frame is None:

            print(
                "[WARNING] Webcam frame "
                "capture failed."
            )

            continue


        frame_counter += 1


        # ------------------------------------------------------------------------------------------
        # Current loop FPS
        # ------------------------------------------------------------------------------------------

        now = time.perf_counter()

        delta = (
            now
            - last_loop_time
        )

        last_loop_time = now


        if delta > 0:

            loop_fps = (
                1.0 / delta
            )

            loop_fps_values.append(
                loop_fps
            )

        else:

            loop_fps = 0.0


        # ------------------------------------------------------------------------------------------
        # Run model only every N webcam frames
        # ------------------------------------------------------------------------------------------

        if (
            frame_counter
            % INFERENCE_EVERY
            == 0
        ):

            prediction_start = (
                time.perf_counter()
            )


            frame_embedding, inference_ms = (
                extract_live_embedding(
                    frame
                )
            )


            inference_latencies.append(
                inference_ms
            )


            embedding_buffer.append(
                frame_embedding
            )


            # --------------------------------------------------------------------------------------
            # Only predict after temporal buffer contains 8 embeddings
            # --------------------------------------------------------------------------------------

            if len(
                embedding_buffer
            ) == TEMPORAL_WINDOW:

                temporal_embedding = (
                    make_temporal_embedding(
                        embedding_buffer
                    )
                )


                if temporal_embedding is not None:

                    (
                        latest_predictions,
                        retrieval_ms
                    ) = retrieve_live_topk(
                        temporal_embedding,
                        TOP_K
                    )


                    retrieval_latencies.append(
                        retrieval_ms
                    )


                    prediction_counter += 1


                    latest_top1 = (
                        latest_predictions[
                            0
                        ][
                            "gloss"
                        ]
                    )


                    latest_similarity = (
                        latest_predictions[
                            0
                        ][
                            "similarity"
                        ]
                    )


                    prediction_ms = (
                        time.perf_counter()
                        - prediction_start
                    ) * 1000.0


                    prediction_latencies.append(
                        prediction_ms
                    )


                    log_row = {

                        "timestamp":
                            datetime.now()
                            .isoformat(),

                        "frame_number":
                            int(
                                frame_counter
                            ),

                        "prediction_number":
                            int(
                                prediction_counter
                            ),

                        "buffer_size":
                            int(
                                len(
                                    embedding_buffer
                                )
                            ),

                        "top1_gloss":
                            str(
                                latest_top1
                            ),

                        "top1_similarity":
                            float(
                                latest_similarity
                            ),

                        "inference_ms":
                            float(
                                inference_ms
                            ),

                        "retrieval_ms":
                            float(
                                retrieval_ms
                            ),

                        "prediction_pipeline_ms":
                            float(
                                prediction_ms
                            ),
                    }


                    for i in range(
                        min(
                            TOP_K,
                            len(
                                latest_predictions
                            )
                        )
                    ):

                        pred = (
                            latest_predictions[
                                i
                            ]
                        )

                        log_row[
                            f"top{i + 1}_gloss"
                        ] = pred[
                            "gloss"
                        ]

                        log_row[
                            f"top{i + 1}_similarity"
                        ] = pred[
                            "similarity"
                        ]


                    log_rows.append(
                        log_row
                    )


        # ------------------------------------------------------------------------------------------
        # DISPLAY
        # ------------------------------------------------------------------------------------------

        display = frame.copy()


        # Semi-transparent information panel
        overlay = display.copy()

        cv2.rectangle(
            overlay,
            (10, 10),
            (560, 335),
            (0, 0, 0),
            -1
        )

        display = cv2.addWeighted(
            overlay,
            0.60,
            display,
            0.40,
            0
        )


        draw_text(
            display,
            "CISLR - RAW OPENVINO",
            (25, 40),
            scale=0.80,
            thickness=2
        )


        draw_text(
            display,
            f"Buffer: "
            f"{len(embedding_buffer)}/"
            f"{TEMPORAL_WINDOW}",
            (25, 72),
            scale=0.60
        )


        draw_text(
            display,
            f"Live FPS: "
            f"{loop_fps:.1f}",
            (25, 102),
            scale=0.60
        )


        if latest_similarity is None:

            draw_text(
                display,
                "Top-1: WARMING UP",
                (25, 145),
                scale=0.75
            )

        else:

            draw_text(
                display,
                f"Top-1: "
                f"{latest_top1}",
                (25, 145),
                scale=0.75
            )

            draw_text(
                display,
                f"Cosine: "
                f"{latest_similarity:.4f}",
                (25, 178),
                scale=0.60
            )


        # ------------------------------------------------------------------------------------------
        # Top-5
        # ------------------------------------------------------------------------------------------

        if latest_predictions:

            draw_text(
                display,
                "Raw Top-5:",
                (25, 215),
                scale=0.62
            )


            y = 245


            for pred in (
                latest_predictions
            ):

                text_line = (
                    f"{pred['rank']}. "
                    f"{pred['gloss']}  "
                    f"{pred['similarity']:.3f}"
                )


                draw_text(
                    display,
                    text_line,
                    (35, y),
                    scale=0.52,
                    thickness=1
                )


                y += 27


        # ------------------------------------------------------------------------------------------
        # Methodological label
        # ------------------------------------------------------------------------------------------

        cv2.putText(
            display,
            "RAW OUTPUT - NO STABILIZATION / NO UNKNOWN REJECTION",
            (
                15,
                actual_height - 20
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        cv2.imshow(
            WINDOW_NAME,
            display
        )


        # ------------------------------------------------------------------------------------------
        # KEYBOARD
        # ------------------------------------------------------------------------------------------

        key = (
            cv2.waitKey(1)
            & 0xFF
        )


        # Q = quit
        if key == ord("q"):

            print(
                "\n[INFO] Q pressed. "
                "Stopping webcam."
            )

            break


        # S = screenshot
        elif key == ord("s"):

            screenshot_counter += 1

            screenshot_path = (
                SCREENSHOT_DIR
                / (
                    f"stage9_5_"
                    f"screenshot_"
                    f"{screenshot_counter:03d}.jpg"
                )
            )


            cv2.imwrite(
                str(
                    screenshot_path
                ),
                display
            )


            print(
                "[SCREENSHOT]",
                screenshot_path
            )


finally:

    cap.release()

    cv2.destroyAllWindows()


# ==================================================================================================
# 12. SESSION METRICS
# ==================================================================================================

session_seconds = (
    time.perf_counter()
    - session_start
)


effective_capture_fps = (
    frame_counter
    / session_seconds
    if session_seconds > 0
    else 0.0
)


prediction_rate = (
    prediction_counter
    / session_seconds
    if session_seconds > 0
    else 0.0
)


def safe_mean(values):

    if not values:
        return None

    return float(
        np.mean(values)
    )


def safe_median(values):

    if not values:
        return None

    return float(
        np.median(values)
    )


def safe_percentile(
    values,
    percentile
):

    if not values:
        return None

    return float(
        np.percentile(
            values,
            percentile
        )
    )


mean_inference_ms = (
    safe_mean(
        inference_latencies
    )
)

median_inference_ms = (
    safe_median(
        inference_latencies
    )
)

p95_inference_ms = (
    safe_percentile(
        inference_latencies,
        95
    )
)


mean_retrieval_ms = (
    safe_mean(
        retrieval_latencies
    )
)

p95_retrieval_ms = (
    safe_percentile(
        retrieval_latencies,
        95
    )
)


mean_prediction_ms = (
    safe_mean(
        prediction_latencies
    )
)

p95_prediction_ms = (
    safe_percentile(
        prediction_latencies,
        95
    )
)


# ==================================================================================================
# 13. SAVE RAW PREDICTION LOG
# ==================================================================================================

LOG_CSV = (
    LOG_DIR
    / "stage9_5_raw_webcam_predictions.csv"
)


if log_rows:

    log_df = pd.DataFrame(
        log_rows
    )

    log_df.to_csv(
        LOG_CSV,
        index=False
    )

else:

    log_df = pd.DataFrame()


# ==================================================================================================
# 14. SUMMARY REPORT
# ==================================================================================================

REPORT_JSON = (
    REPORT_DIR
    / "stage9_5_raw_webcam_summary.json"
)


report = {

    "stage":
        "9.5",

    "name":
        "Raw Live Webcam OpenVINO Recognition",

    "engineering_status":
        (
            "PASS"
            if (
                frame_counter > 0
                and prediction_counter > 0
            )
            else "INCOMPLETE"
        ),

    "camera_index":
        int(
            CAMERA_INDEX
        ),

    "camera_resolution":
        [
            int(actual_width),
            int(actual_height),
        ],

    "reported_camera_fps":
        float(
            actual_camera_fps
        ),

    "openvino_device":
        OPENVINO_DEVICE,

    "prototype_count":
        int(
            len(
                prototype_bank
            )
        ),

    "embedding_dimension":
        960,

    "temporal_window":
        int(
            TEMPORAL_WINDOW
        ),

    "inference_every_n_frames":
        int(
            INFERENCE_EVERY
        ),

    "session_seconds":
        float(
            session_seconds
        ),

    "captured_frames":
        int(
            frame_counter
        ),

    "predictions_generated":
        int(
            prediction_counter
        ),

    "effective_capture_fps":
        float(
            effective_capture_fps
        ),

    "prediction_rate_hz":
        float(
            prediction_rate
        ),

    "mean_openvino_inference_ms":
        mean_inference_ms,

    "median_openvino_inference_ms":
        median_inference_ms,

    "p95_openvino_inference_ms":
        p95_inference_ms,

    "mean_retrieval_ms":
        mean_retrieval_ms,

    "p95_retrieval_ms":
        p95_retrieval_ms,

    "mean_prediction_pipeline_ms":
        mean_prediction_ms,

    "p95_prediction_pipeline_ms":
        p95_prediction_ms,

    "screenshots_saved":
        int(
            screenshot_counter
        ),

    "stabilization_enabled":
        False,

    "unknown_rejection_enabled":
        False,

    "methodology": {

        "input":
            "live webcam",

        "spatial_input":
            "full frame",

        "spatial_model":
            (
                "OpenVINO "
                "MobileNetV3-Large "
                "960-D extractor"
            ),

        "temporal_representation":
            (
                "rolling 8-embedding "
                "mean pool"
            ),

        "normalization":
            "L2",

        "retrieval":
            "cosine similarity",

        "gallery":
            (
                "frozen Stage 4.5 "
                "canonical prototype gallery"
            ),

        "hand_roi":
            False,

        "tta":
            False,

        "whitening":
            False,

        "csls":
            False,

        "prediction_stabilization":
            False,

        "test_label_tuning":
            False,
    },

    "scientific_note":
        (
            "Stage 9.5 measures raw live "
            "deployment behaviour. Webcam "
            "predictions do not constitute "
            "official CISLR accuracy because "
            "ground-truth webcam labels are "
            "not available."
        ),
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


# ==================================================================================================
# 15. FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("STAGE 9.5 — SESSION SUMMARY")
print("=" * 100)


print(
    f"Session duration          : "
    f"{session_seconds:.2f} s"
)

print(
    f"Captured frames           : "
    f"{frame_counter}"
)

print(
    f"Raw predictions generated : "
    f"{prediction_counter}"
)

print(
    f"Effective capture FPS     : "
    f"{effective_capture_fps:.2f}"
)

print(
    f"Prediction rate           : "
    f"{prediction_rate:.2f} Hz"
)


if mean_inference_ms is not None:

    print(
        f"Mean OV inference         : "
        f"{mean_inference_ms:.3f} ms"
    )

    print(
        f"Median OV inference       : "
        f"{median_inference_ms:.3f} ms"
    )

    print(
        f"P95 OV inference          : "
        f"{p95_inference_ms:.3f} ms"
    )


if mean_retrieval_ms is not None:

    print(
        f"Mean retrieval            : "
        f"{mean_retrieval_ms:.3f} ms"
    )

    print(
        f"P95 retrieval             : "
        f"{p95_retrieval_ms:.3f} ms"
    )


if mean_prediction_ms is not None:

    print(
        f"Mean prediction pipeline  : "
        f"{mean_prediction_ms:.3f} ms"
    )

    print(
        f"P95 prediction pipeline   : "
        f"{p95_prediction_ms:.3f} ms"
    )


print(
    f"Screenshots saved         : "
    f"{screenshot_counter}"
)


print("\nSaved:")

if log_rows:
    print(LOG_CSV)
else:
    print(
        "[WARNING] No prediction log "
        "was generated."
    )

print(REPORT_JSON)


print("\n" + "-" * 100)


if (
    frame_counter > 0
    and prediction_counter > 0
):

    print(
        "STAGE 9.5 — ENGINEERING PASS"
    )

    print(
        "\nLive webcam frames were "
        "successfully processed by the "
        "frozen OpenVINO retrieval engine."
    )

    print(
        "\nIMPORTANT:"
        "\nThe displayed predictions are RAW."
        "\nPrediction changes, false positives "
        "and predictions while no sign is being "
        "performed are expected at this stage."
    )

    print(
        "\nNEXT:"
        "\nInspect raw webcam behaviour before "
        "designing Stage 9.6 stabilization "
        "and UNKNOWN rejection."
    )

else:

    print(
        "STAGE 9.5 — INCOMPLETE"
    )

    print(
        "\nNo usable live predictions "
        "were generated."
    )


print("=" * 100)

CISLR — STAGE 9.5
RAW LIVE WEBCAM OPENVINO RECOGNITION

ARTIFACT CHECK
----------------------------------------------------------------------------------------------------
OpenVINO model           : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage9_final_openvino/embedding_fp32/mobilenet_v3_large_embedding_960_fp32_static.xml
                           Exists = True
Prototype embeddings     : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_embeddings.npy
                           Exists = True
Prototype metadata       : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_metadata.csv
                           Exists = True

[OK] Required artifacts exist.

PROTOTYPE GALLERY
Prototype count : 4764
Embedding dim   : 960
Unique glosses  : 4764

[OK] Gallery ready.

OPENVINO
Device      : CPU
Input shape : (1, 3, 224, 224)
Output shape: (1, 960)

[OK] OpenV